In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers accelerate qwen-vl-utils tqdm

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 28.0 MB/s eta 0:00:00


In [2]:
import os; os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import os, json, re, torch
from PIL import Image
from tqdm import tqdm

print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

ROOT = '/content/drive/MyDrive/Colab Notebooks/BYTE ML Task Data'
OUT = '/content/drive/MyDrive/manga_outputs'
os.makedirs(OUT, exist_ok=True)

seqs = json.load(open(f'{ROOT}/sequences.json'))
dev = [s for s in seqs if s['split'] == 'development']
test = [s for s in seqs if s['split'] == 'test']
labels = {}
for line in open(f'{ROOT}/development/labels.jsonl'):
    r = json.loads(line)
    labels[r['sequence_id']] = r['pages']
print(len(dev), len(test), len(labels))

GPU: True Tesla T4
80 15 80


In [4]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
name = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(name, torch_dtype=torch.float16, device_map="cuda")
processor = AutoProcessor.from_pretrained(name, max_pixels=1300*28*28)
print("loaded")

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

loaded


In [5]:
PROMPT = """Read this manga page panel by panel, right to left.
List every speech balloon and thought balloon's text, in the order you'd read them.
Do NOT include sound effects like "SNAP" or "BOOM" written directly on the art without a balloon.

Format your answer EXACTLY like this example, one line per balloon:
character1: Where are you going?
character2: Home.
NARRATION: Later that day...

Now do it for this page:"""

def get_page_text(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    result = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    del inputs, out
    torch.cuda.empty_cache()
    return result

def parse_lines(raw):
    lines = []
    for line in raw.strip().split('\n'):
        m = re.match(r'^([A-Za-z0-9_]+):\s*(.+)$', line.strip())
        if m:
            lines.append({"speaker": m.group(1), "text": m.group(2).strip()})
    return lines

print("functions ready")

functions ready


In [5]:
img = Image.open(f"{ROOT}/{dev[0]['images'][0]}")
raw = get_page_text(img)
print(raw)
print(parse_lines(raw))

/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


A STRONG ARMADA MEAN FAST BALL!
THIS IS DARAYOUTH GRAFFITI!
HE'S DOING REALLY WELL.
IT LOOKS LIKE HE'S BEEN WORKING HARD FOR THE FALL TOURNAMENT.
HUUH? DOES KOMABA SEEM DIFFERENT THAN USUAL?
OUT FOR BLOOD.
NO, THAT'S NOT IT. IT'S LIKE HE'S...
[]


In [6]:
def parse_lines(raw):
    lines = []
    for line in raw.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        m = re.match(r'^([A-Za-z0-9_]+):\s*(.+)$', line)
        if m:
            lines.append({"speaker": m.group(1), "text": m.group(2).strip()})
        else:
            lines.append({"speaker": "character1", "text": line})
    return lines

N = 20
PRED = f'{OUT}/baseline_dev.jsonl'

rows, done = [], set()
if os.path.exists(PRED):
    for line in open(PRED):
        r = json.loads(line); rows.append(r); done.add(r['sequence_id'])
print("already done:", len(done))

for seq in tqdm(dev[:N]):
    sid = seq['sequence_id']
    if sid in done:
        continue
    pages = []
    try:
        for p in seq['images']:
            raw = get_page_text(Image.open(f"{ROOT}/{p}"))
            pages.append(parse_lines(raw))
    except Exception as e:
        print("STOP, error on", sid, ":", str(e)[:200])
        break
    rows.append({"sequence_id": sid, "pages": pages})
    with open(PRED, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')

print("saved:", len(rows))

already done: 0


100%|██████████| 20/20 [10:00<00:00, 30.00s/it]

saved: 20


In [7]:
  ids = [s['sequence_id'] for s in dev[:20]]

REF20 = f'{OUT}/labels_first20.jsonl'
with open(REF20, 'w') as f:
    for sid in ids:
        f.write(json.dumps({"sequence_id": sid, "pages": labels[sid]}) + '\n')

!python "{ROOT}/score.py" --references "{REF20}" --predictions "{OUT}/baseline_dev.jsonl" --output "{OUT}/baseline_scores.json"

{
  "text_order_score": 0.4884733190749461,
  "balanced_joint_f1": 0.1537427325444578,
  "speaker_accuracy_on_matched": 0.5111491424030591,
  "joint_f1": 0.301248645122831,
  "matched_token_coverage": 0.6180452675232402
}


In [8]:
pred = {}
for line in open(f'{OUT}/baseline_dev.jsonl'):
    r = json.loads(line); pred[r['sequence_id']] = r['pages']

for sid in ids[:3]:
    for pg in range(3):
        print("=====", sid, "page", pg+1)
        print("GOLD:")
        for l in labels[sid][pg]:
            print("  ", l['speaker'], "|", l['text'])
        print("PRED:")
        for l in pred[sid][pg]:
            print("  ", l['speaker'], "|", l['text'])

===== seq_2032620aa4e4ac7f page 1
GOLD:
   character1 | HUH? DOES KOMABA SEEM DIFFERENT THAN USUAL?
   character2 | HE'S DOING REALLY WELL. IT LOOKS LIKE HE'S BEEN WORKING HARD FOR THE FALL TOURNAMENT.
   UNKNOWN | THAT'S THE WAY!
   character1 | NO, THAT'S NOT IT. IT'S LIKE HE'S...
   character1 | OUT FOR BLOOD.
PRED:
   character1 | A STRONG ARMADA MEAN FAST BALL!
   character1 | THIS IS DARAYOUTH GRAFFITI!
   character1 | HE'S DOING REALLY WELL.
   character1 | IT LOOKS LIKE HE'S BEEN WORKING HARD FOR THE FALL TOURNAMENT.
   character1 | HUUH? DOES KOMABA SEEM DIFFERENT THAN USUAL?
   character1 | OUT FOR BLOOD.
   character1 | NO, THAT'S NOT IT. IT'S LIKE HE'S...
===== seq_2032620aa4e4ac7f page 2
GOLD:
PRED:
   SFX | HOOFBEATS
   Hachiken | Last time...
   Hachiken | Summer vacation... Hachiken spent the whole time working at Art's, the new term began... and Hachiken used the money he earned to buy Pork Bowl 5, 5 Meat. But he's still not sure of his choices. Now, the seasons change

In [ ]:
pred = {}
for line in open(f'{OUT}/baseline_dev.jsonl'):
    r = json.loads(line); pred[r['sequence_id']] = r['pages']

for sid in ids:
    g = {l['speaker'] for p in labels[sid] for l in p}
    p_ = {l['speaker'] for p in pred[sid] for l in p}
    print(sid[:12], "| gold speakers:", len(g), "| pred speakers:", len(p_))

In [6]:
def to_target(page_lines):
    out = []
    for l in page_lines:
        spk = 'character1' if l['speaker'] == 'UNKNOWN' else l['speaker']
        out.append(f"{spk}: {l['text']}")
    return "\n".join(out) if out else "NONE"

val_ids = [s['sequence_id'] for s in dev[:20]]
train_seqs = dev[20:]

examples = []
for s in train_seqs:
    for pg, p in enumerate(s['images']):
        examples.append({
            "image_path": f"{ROOT}/{p}",
            "target": to_target(labels[s['sequence_id']][pg])
        })

print("train examples:", len(examples))
print("empty pages:", sum(e['target'] == "NONE" for e in examples))
print(examples[0]['target'][:300])

train examples: 180
empty pages: 9
character1: POCHITA... I'LL PROBABLY GET MYSELF KILLED FIGHTING DEVILS BEFORE LONG.


In [7]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
!pip install -q peft
import gc
from peft import LoraConfig, get_peft_model

cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, cfg)
model.print_trainable_parameters()

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False
model.train()

for n, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()

def build_example(ex):
    img = Image.open(ex["image_path"])
    user = [{"role": "user", "content": [
        {"type": "image", "image": img},
        {"type": "text", "text": PROMPT}]}]
    full = user + [{"role": "assistant", "content": [{"type": "text", "text": ex["target"]}]}]
    prompt_text = processor.apply_chat_template(user, tokenize=False, add_generation_prompt=True)
    full_text = processor.apply_chat_template(full, tokenize=False)
    prompt_len = processor(text=[prompt_text], images=[img], return_tensors="pt")["input_ids"].shape[1]
    enc = processor(text=[full_text], images=[img], return_tensors="pt")
    labels_ = enc["input_ids"].clone()
    labels_[:, :prompt_len] = -100
    enc["labels"] = labels_
    return enc

enc = build_example(examples[0]).to("cuda")
print("seq length:", enc["input_ids"].shape[1])
with torch.autocast("cuda", dtype=torch.float16):
    loss = model(**enc).loss
loss.backward()
print("loss:", loss.item())
print("max GPU memory (GB):", torch.cuda.max_memory_allocated() / 1e9)
model.zero_grad(set_to_none=True)
del enc, loss
gc.collect(); torch.cuda.empty_cache()

trainable params: 7,372,800 || all params: 3,761,995,776 || trainable%: 0.1960


/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


seq length: 1394


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


loss: 0.617525041103363
max GPU memory (GB): 10.799736832


In [9]:
import random, time
from torch.optim import AdamW

ADAPTER_DIR = f'{OUT}/lora_adapter'
params = [p for p in model.parameters() if p.requires_grad]
opt = AdamW(params, lr=2e-4, weight_decay=0.0)
scaler = torch.amp.GradScaler("cuda")
ACCUM = 8
EPOCHS = 1

random.seed(0)
model.train()
recent, skipped = [], 0
t0 = time.time()

for ep in range(EPOCHS):
    order = list(range(len(examples)))
    random.shuffle(order)
    for i, idx in enumerate(tqdm(order)):
        enc = loss = None
        try:
            enc = build_example(examples[idx]).to("cuda")
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**enc).loss
            if torch.isfinite(loss):
                scaler.scale(loss / ACCUM).backward()
                recent.append(loss.item())
            else:
                skipped += 1
        except torch.cuda.OutOfMemoryError:
            skipped += 1
            opt.zero_grad(set_to_none=True)
        enc = loss = None

        if (i + 1) % ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

        if (i + 1) % 20 == 0:
            print(f"ep{ep} ex{i+1}: avg loss {sum(recent)/len(recent):.3f} | skipped {skipped} | {(time.time()-t0)/60:.1f} min")
            recent = []
            torch.cuda.empty_cache()
        if (i + 1) % 40 == 0:
            model.save_pretrained(ADAPTER_DIR)

model.save_pretrained(ADAPTER_DIR)
print("training done, adapter saved to", ADAPTER_DIR)



  0%|          | 0/180 [00:00<?, ?it/s]

  1%|          | 1/180 [00:10<30:30, 10.22s/it]

  1%|          | 2/180 [00:17<25:35,  8.62s/it]

  2%|▏         | 3/180 [00:24<23:05,  7.83s/it]

  2%|▏         | 4/180 [00:31<22:01,  7.51s/it]

  3%|▎         | 5/180 [00:38<21:29,  7.37s/it]

  3%|▎         | 6/180 [00:45<20:47,  7.17s/it]

  4%|▍         | 7/180 [00:52<20:00,  6.94s/it]

  4%|▍         | 8/180 [00:58<19:46,  6.90s/it]

  5%|▌         | 9/180 [01:04<18:54,  6.63s/it]

  6%|▌         | 10/180 [01:11<18:53,  6.67s/it]

  6%|▌         | 11/180 [01:18<19:05,  6.78s/it]

  7%|▋         | 12/180 [01:25<19:28,  6.95s/it]

  7%|▋         | 13/180 [01:32<18:58,  6.82s/it]

  8%|▊         | 14/180 [01:39<19:12,  6.94s/it]

  8%|▊         | 15/180 [01:46<18:43,  6.81s/it]

  9%|▉         | 16/180 [01:52<18:31,  6.78s/it]

  9%|▉         | 17/180 [02:00<19:15,  7.09s/it]

 10%|█         | 18/180 [02:08<19:17,  7.14s/it]

 11%|█         | 19/180 [02:15<19:09,  7.14s/it]

 11%|█         |

ep0 ex20: avg loss 0.643 | skipped 0 | 2.4 min




 12%|█▏        | 21/180 [02:28<18:39,  7.04s/it]

 12%|█▏        | 22/180 [02:35<18:20,  6.97s/it]

 13%|█▎        | 23/180 [02:42<17:56,  6.85s/it]

 13%|█▎        | 24/180 [02:49<17:42,  6.81s/it]

 14%|█▍        | 25/180 [02:55<17:29,  6.77s/it]

 14%|█▍        | 26/180 [03:02<17:17,  6.74s/it]

 15%|█▌        | 27/180 [03:10<17:55,  7.03s/it]

 16%|█▌        | 28/180 [03:16<17:28,  6.90s/it]

 16%|█▌        | 29/180 [03:23<17:17,  6.87s/it]

 17%|█▋        | 30/180 [03:30<17:00,  6.80s/it]

 17%|█▋        | 31/180 [03:37<17:19,  6.98s/it]

 18%|█▊        | 32/180 [03:44<16:58,  6.88s/it]

 18%|█▊        | 33/180 [03:50<16:37,  6.78s/it]

 19%|█▉        | 34/180 [03:57<16:36,  6.83s/it]

 19%|█▉        | 35/180 [04:04<16:30,  6.83s/it]

 20%|██        | 36/180 [04:11<16:28,  6.87s/it]

 21%|██        | 37/180 [04:19<17:02,  7.15s/it]

 21%|██        | 38/180 [04:25<16:23,  6.93s/it]

 22%|██▏       | 39/180 [04:32<16:02,  6.82s/it]

ep0 ex40: avg loss 0.499 | skipped 0 | 4.7 min




 22%|██▏       | 40/180 [04:39<16:16,  6.98s/it]

 23%|██▎       | 41/180 [04:47<16:41,  7.20s/it]

 23%|██▎       | 42/180 [04:53<15:52,  6.90s/it]

 24%|██▍       | 43/180 [05:00<15:43,  6.89s/it]

 24%|██▍       | 44/180 [05:06<15:22,  6.78s/it]

 25%|██▌       | 45/180 [05:15<16:27,  7.32s/it]

 26%|██▌       | 46/180 [05:22<15:54,  7.13s/it]

 26%|██▌       | 47/180 [05:28<15:34,  7.03s/it]

 27%|██▋       | 48/180 [05:35<14:58,  6.81s/it]

 27%|██▋       | 49/180 [05:41<14:37,  6.70s/it]

 28%|██▊       | 50/180 [05:44<11:59,  5.53s/it]

 28%|██▊       | 51/180 [05:51<12:41,  5.91s/it]

 29%|██▉       | 52/180 [05:58<13:12,  6.19s/it]

 29%|██▉       | 53/180 [06:04<13:19,  6.29s/it]

 30%|███       | 54/180 [06:11<13:36,  6.48s/it]

 31%|███       | 55/180 [06:18<13:32,  6.50s/it]

 31%|███       | 56/180 [06:24<13:33,  6.56s/it]

 32%|███▏      | 57/180 [06:31<13:22,  6.52s/it]

 32%|███▏      | 58/180 [06:38<13:49,  6.80s/it]

 33%|███▎      | 59/180 [06:46<14:04,  6.98s/it]

...



 78%|███████▊  | 141/180 [16:29<04:44,  7.29s/it]

 79%|███████▉  | 142/180 [16:36<04:27,  7.05s/it]

 79%|███████▉  | 143/180 [16:43<04:18,  7.00s/it]

 80%|████████  | 144/180 [16:49<04:07,  6.86s/it]

 81%|████████  | 145/180 [16:56<04:01,  6.91s/it]

 81%|████████  | 146/180 [17:03<03:51,  6.81s/it]

 82%|████████▏ | 147/180 [17:10<03:49,  6.95s/it]

 82%|████████▏ | 148/180 [17:17<03:39,  6.87s/it]

 83%|████████▎ | 149/180 [17:24<03:36,  7.00s/it]

 83%|████████▎ | 150/180 [17:30<03:24,  6.83s/it]

 84%|████████▍ | 151/180 [17:37<03:18,  6.86s/it]

 84%|████████▍ | 152/180 [17:44<03:11,  6.85s/it]

 85%|████████▌ | 153/180 [17:52<03:08,  7.00s/it]

 86%|████████▌ | 154/180 [17:59<03:01,  6.99s/it]

 86%|████████▌ | 155/180 [18:06<02:57,  7.09s/it]

 87%|████████▋ | 156/180 [18:12<02:44,  6.87s/it]

 87%|████████▋ | 157/180 [18:19<02:36,  6.82s/it]

 88%|████████▊ | 158/180 [18:26<02:31,  6.89s/it]

 88%|████████▊ | 159/180 [18:32<02:22,  6.77s/it]

ep0 ex160: avg loss 0.208 | skipped 1 | 18.7 min




 89%|████████▉ | 160/180 [18:40<02:17,  6.86s/it]

 89%|████████▉ | 161/180 [18:47<02:16,  7.18s/it]

 90%|█████████ | 162/180 [18:54<02:06,  7.02s/it]

 91%|█████████ | 163/180 [19:01<01:57,  6.89s/it]

 91%|█████████ | 164/180 [19:07<01:48,  6.78s/it]

 92%|█████████▏| 165/180 [19:14<01:41,  6.76s/it]

 92%|█████████▏| 166/180 [19:21<01:36,  6.89s/it]

 93%|█████████▎| 167/180 [19:28<01:28,  6.79s/it]

 93%|█████████▎| 168/180 [19:35<01:21,  6.80s/it]

 94%|█████████▍| 169/180 [19:41<01:13,  6.69s/it]

 94%|█████████▍| 170/180 [19:48<01:09,  6.91s/it]

 95%|█████████▌| 171/180 [19:55<01:01,  6.86s/it]

 96%|█████████▌| 172/180 [20:02<00:54,  6.78s/it]

 96%|█████████▌| 173/180 [20:09<00:48,  6.99s/it]

 97%|█████████▋| 174/180 [20:16<00:41,  6.93s/it]

 97%|█████████▋| 175/180 [20:23<00:34,  6.99s/it]

 98%|█████████▊| 176/180 [20:30<00:27,  6.89s/it]

 98%|█████████▊| 177/180 [20:37<00:20,  6.92s/it]

 99%|█████████▉| 178/180 [20:43<00:13,  6.73s/it]

 99%|█████████▉| 179/180 [20:

ep0 ex180: avg loss 0.239 | skipped 1 | 21.0 min


training done, adapter saved to /content/drive/MyDrive/manga_outputs/lora_adapter


In [10]:
model.eval()
model.config.use_cache = True

def get_page_text_ft(image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        out = model.generate(**inputs, max_new_tokens=500, do_sample=False, use_cache=True)
    result = processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    del inputs, out
    torch.cuda.empty_cache()
    return result

def parse_ft(raw):
    raw = raw.strip()
    if raw == "NONE" or not raw:
        return []
    lines = []
    for line in raw.split('\n'):
        line = line.strip()
        if not line or line == "NONE":
            continue
        m = re.match(r'^([A-Za-z0-9_]+):\s*(.+)$', line)
        if m:
            lines.append({"speaker": m.group(1), "text": m.group(2).strip()})
        else:
            lines.append({"speaker": "character1", "text": line})
    return lines

PRED_FT = f'{OUT}/finetuned_dev.jsonl'
rows, done = [], set()
if os.path.exists(PRED_FT):
    for line in open(PRED_FT):
        r = json.loads(line); rows.append(r); done.add(r['sequence_id'])
print("already done:", len(done))

for seq in tqdm(dev[:20]):
    sid = seq['sequence_id']
    if sid in done:
        continue
    pages = []
    try:
        for p in seq['images']:
            pages.append(parse_ft(get_page_text_ft(Image.open(f"{ROOT}/{p}"))))
    except Exception as e:
        print("STOP, error on", sid, ":", str(e)[:200])
        break
    rows.append({"sequence_id": sid, "pages": pages})
    with open(PRED_FT, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')

print("saved:", len(rows))

already done: 0




  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


  5%|▌         | 1/20 [00:36<11:40, 36.86s/it]

 10%|█         | 2/20 [01:35<14:55, 49.72s/it]

 15%|█▌        | 3/20 [02:39<15:53, 56.11s/it]

 20%|██        | 4/20 [03:34<14:53, 55.85s/it]

 25%|██▌       | 5/20 [03:59<11:09, 44.64s/it]

 30%|███       | 6/20 [04:53<11:11, 47.99s/it]

 35%|███▌      | 7/20 [05:40<10:18, 47.55s/it]

 40%|████      | 8/20 [06:39<10:14, 51.18s/it]

 45%|████▌     | 9/20 [07:53<10:42, 58.44s/it]

 50%|█████     | 10/20 [08:39<09:03, 54.31s/it]

 55%|█████▌    | 11/20 [09:24<07:44, 51.57s/it]

 60%|██████    | 12/20 [09:58<06:09, 46.22s/it]

 65%|██████▌   | 13/20 [10:44<05:23, 46.20s/it]

 70%|

saved: 20


In [11]:
!python "{ROOT}/score.py" --references "{OUT}/labels_first20.jsonl" --predictions "{OUT}/finetuned_dev.jsonl" --output "{OUT}/finetuned_scores.json"

{
  "text_order_score": 0.6645035695467771,
  "balanced_joint_f1": 0.16745484806612915,
  "speaker_accuracy_on_matched": 0.39864135180216714,
  "joint_f1": 0.30358831556409493,
  "matched_token_coverage": 0.7590973866584486
}


In [12]:
sub = [json.loads(l) for l in open(f'{ROOT}/sample_submission.jsonl')]
test_map = {s['sequence_id']: s for s in test}
PRED_TEST = f'{OUT}/test_predictions.jsonl'

rows, done = [], set()
if os.path.exists(PRED_TEST):
    for line in open(PRED_TEST):
        r = json.loads(line); rows.append(r); done.add(r['sequence_id'])
print("already done:", len(done), "of", len(sub))

for row in tqdm(sub):
    sid = row['sequence_id']
    if sid in done:
        continue
    pages = []
    try:
        for p in test_map[sid]['images']:
            pages.append(parse_ft(get_page_text_ft(Image.open(f"{ROOT}/{p}"))))
    except Exception as e:
        print("STOP, error on", sid, ":", str(e)[:200])
        break
    rows.append({"sequence_id": sid, "pages": pages})
    with open(PRED_TEST, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')

# format check
ok = len(rows) == len(sub)
for r in rows:
    ok &= len(r['pages']) == 3
    for pg in r['pages']:
        for l in pg:
            ok &= set(l) == {"speaker", "text"} and bool(l['text'].strip())
print("sequences:", len(rows), "| format OK:", ok)

already done: 0 of 15




  0%|          | 0/15 [00:00<?, ?it/s]

  7%|▋         | 1/15 [01:32<21:30, 92.20s/it]

 13%|█▎        | 2/15 [02:58<19:12, 88.63s/it]

 20%|██        | 3/15 [03:31<12:40, 63.35s/it]

 27%|██▋       | 4/15 [04:08<09:44, 53.11s/it]

 33%|███▎      | 5/15 [04:20<06:22, 38.23s/it]

 40%|████      | 6/15 [04:33<04:24, 29.36s/it]

 47%|████▋     | 7/15 [05:34<05:18, 39.78s/it]

 53%|█████▎    | 8/15 [06:43<05:43, 49.12s/it]

 60%|██████    | 9/15 [08:13<06:12, 62.10s/it]

 67%|██████▋   | 10/15 [09:09<05:00, 60.12s/it]

 73%|███████▎  | 11/15 [10:11<04:02, 60.65s/it]

 80%|████████  | 12/15 [11:01<02:52, 57.40s/it]

 87%|████████▋ | 13/15 [11:27<01:35, 47.91s/it]

 93%|█████████▎| 14/15 [11:44<00:38, 38.64s/it]

100%|██████████| 15/15 [11:59<00:00, 47.99s/it]

sequences: 15 | format OK: True


In [13]:
sub = [json.loads(l) for l in open(f'{ROOT}/sample_submission.jsonl')]
test_map = {s['sequence_id']: s for s in test}
PRED_TEST = f'{OUT}/test_predictions.jsonl'

rows, done = [], set()
if os.path.exists(PRED_TEST):
    for line in open(PRED_TEST):
        r = json.loads(line); rows.append(r); done.add(r['sequence_id'])
print("already done:", len(done), "of", len(sub))

for row in tqdm(sub):
    sid = row['sequence_id']
    if sid in done:
        continue
    pages = []
    try:
        for p in test_map[sid]['images']:
            pages.append(parse_ft(get_page_text_ft(Image.open(f"{ROOT}/{p}"))))
    except Exception as e:
        print("STOP, error on", sid, ":", str(e)[:200])
        break
    rows.append({"sequence_id": sid, "pages": pages})
    with open(PRED_TEST, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')

# format check
ok = len(rows) == len(sub)
for r in rows:
    ok &= len(r['pages']) == 3
    for pg in r['pages']:
        for l in pg:
            ok &= set(l) == {"speaker", "text"} and bool(l['text'].strip())
print("sequences:", len(rows), "| format OK:", ok)

already done: 15 of 15




100%|██████████| 15/15 [00:00<00:00, 100342.20it/s]

sequences: 15 | format OK: True


In [14]:
from collections import Counter
import subprocess

def limit_speakers(pages, K):
    cnt = Counter(l['speaker'] for p in pages for l in p if l['speaker'] != 'NARRATION')
    if not cnt:
        return pages
    keep = {s for s, _ in cnt.most_common(K)}
    top = cnt.most_common(1)[0][0]
    return [[{"speaker": (l['speaker'] if (l['speaker'] == 'NARRATION' or l['speaker'] in keep) else top),
              "text": l['text']} for l in p] for p in pages]

REF = f'{OUT}/labels_first20.jsonl'
ft = [json.loads(l) for l in open(f'{OUT}/finetuned_dev.jsonl')]

def score_rows(rows, tag):
    path = f'{OUT}/{tag}.jsonl'
    with open(path, 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')
    res = subprocess.run(["python", f"{ROOT}/score.py", "--references", REF,
                          "--predictions", path, "--output", f"{OUT}/{tag}_scores.json"],
                         capture_output=True, text=True)
    return json.loads(res.stdout)

results = {}
m = score_rows(ft, "post_none")
results[0] = m['balanced_joint_f1']
print("no limit | text_order:", round(m['text_order_score'], 3), "| bal_f1:", round(m['balanced_joint_f1'], 3))

for K in [1, 2, 3, 4, 5, 8]:
    rows = [{"sequence_id": r['sequence_id'], "pages": limit_speakers(r['pages'], K)} for r in ft]
    m = score_rows(rows, f"post_K{K}")
    results[K] = m['balanced_joint_f1']
    print("K =", K, "| text_order:", round(m['text_order_score'], 3), "| bal_f1:", round(m['balanced_joint_f1'], 3))

best = max(results, key=results.get)
print("BEST K:", best, "(0 = no limit)")

test_rows = [json.loads(l) for l in open(f'{OUT}/test_predictions.jsonl')]
if best != 0:
    test_rows = [{"sequence_id": r['sequence_id'], "pages": limit_speakers(r['pages'], best)} for r in test_rows]

sub = [json.loads(l) for l in open(f'{ROOT}/sample_submission.jsonl')]
by_id = {r['sequence_id']: r for r in test_rows}
final = [by_id[s['sequence_id']] for s in sub]
with open(f'{OUT}/final_predictions.jsonl', 'w') as f:
    for r in final:
        f.write(json.dumps(r) + '\n')

ok = len(final) == 15 and all(len(r['pages']) == 3 for r in final)
print("final rows:", len(final), "| format OK:", ok)

no limit | text_order: 0.665 | bal_f1: 0.167
K = 1 | text_order: 0.665 | bal_f1: 0.19
K = 2 | text_order: 0.665 | bal_f1: 0.279
K = 3 | text_order: 0.665 | bal_f1: 0.244
K = 4 | text_order: 0.665 | bal_f1: 0.223
K = 5 | text_order: 0.665 | bal_f1: 0.217
K = 8 | text_order: 0.665 | bal_f1: 0.185
BEST K: 2 (0 = no limit)
final rows: 15 | format OK: True


In [15]:
import shutil
PKG = '/content/drive/MyDrive/manga_submission'
os.makedirs(PKG, exist_ok=True)

shutil.copy(f'{OUT}/final_predictions.jsonl', f'{PKG}/predictions.jsonl')
shutil.copytree(f'{OUT}/lora_adapter', f'{PKG}/lora_adapter', dirs_exist_ok=True)
for name in ['baseline_dev.jsonl', 'finetuned_dev.jsonl', 'baseline_scores.json',
             'finetuned_scores.json', 'labels_first20.jsonl']:
    if os.path.exists(f'{OUT}/{name}'):
        shutil.copy(f'{OUT}/{name}', f'{PKG}/{name}')

for root, _, files in os.walk(PKG):
    for fn in files:
        p = os.path.join(root, fn)
        print(p.replace(PKG, ''), round(os.path.getsize(p) / 1e6, 2), "MB")

/predictions.jsonl 0.02 MB
/baseline_dev.jsonl 0.05 MB
/finetuned_dev.jsonl 0.04 MB
/baseline_scores.json 0.02 MB
/finetuned_scores.json 0.02 MB
/labels_first20.jsonl 0.03 MB
/lora_adapter/README.md 0.01 MB
/lora_adapter/adapter_model.safetensors 29.53 MB
/lora_adapter/adapter_config.json 0.0 MB


In [16]:
import transformers, peft
open(f'{PKG}/requirements.txt', 'w').write(
    f"transformers=={transformers.__version__}\npeft=={peft.__version__}\n"
    f"torch=={torch.__version__}\naccelerate\nqwen-vl-utils\ntqdm\npillow\n")
print(open(f'{PKG}/requirements.txt').read())

transformers==5.18.0
peft==0.21.1
torch==2.11.0+cu130
accelerate
qwen-vl-utils
tqdm
pillow



In [ ]:
import importlib.util, json, os, shutil

# 1. notebook ki copy package mein
try:
    from google.colab import _message
    nb = _message.blocking_request('get_ipynb', timeout_sec=120)['ipynb']
    json.dump(nb, open(f'{PKG}/BYTE_TASK_notebook.ipynb', 'w'))
    print("notebook saved")
except Exception as e:
    print("notebook auto-save failed:", e, "-> File > Download > .ipynb se haath se download karo")

# 2. config aur sab scores ek file mein
def macro(path):
    return json.load(open(path))['macro']

results = {
    "base_model": "Qwen/Qwen2.5-VL-3B-Instruct",
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05,
             "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]},
    "training": {"epochs": 1, "lr": 2e-4, "grad_accum": 8, "train_sequences": 60,
                 "train_pages": 180, "val_sequences": 20, "skipped_examples": 1,
                 "final_avg_loss": 0.239, "minutes": 21},
    "image_max_pixels": 1300 * 28 * 28,
    "decoding": {"do_sample": False, "max_new_tokens_baseline": 400, "max_new_tokens_finetuned": 500},
    "scores_first20_dev": {
        "baseline": macro(f'{OUT}/baseline_scores.json'),
        "finetuned": macro(f'{OUT}/finetuned_scores.json'),
        "speaker_limit_sweep": {},
    },
    "chosen_K": 2,
}
for tag in ["post_none", "post_K1", "post_K2", "post_K3", "post_K4", "post_K5", "post_K8"]:
    p = f'{OUT}/{tag}_scores.json'
    if os.path.exists(p):
        results["scores_first20_dev"]["speaker_limit_sweep"][tag] = macro(p)
json.dump(results, open(f'{PKG}/results.json', 'w'), indent=2)

# 3. speaker limit se pehle wali raw test predictions bhi rakho
shutil.copy(f'{OUT}/test_predictions.jsonl', f'{PKG}/test_predictions_raw_before_speaker_limit.jsonl')

# 4. score.py ke apne loader se final file check
spec = importlib.util.spec_from_file_location("score", f"{ROOT}/score.py")
sc = importlib.util.module_from_spec(spec); spec.loader.exec_module(sc)
preds = sc.load_jsonl(f'{PKG}/predictions.jsonl')
expected = [json.loads(l)['sequence_id'] for l in open(f'{ROOT}/sample_submission.jsonl')]
print("scorer loader OK:", len(preds), "| ids/order match sample:", list(preds) == expected)

# 5. sanity stats (gold mein average ~7.3 lines/page tha)
n_lines = [len(pg) for pages in preds.values() for pg in pages]
print("avg lines/page:", round(sum(n_lines) / len(n_lines), 2),
      "| empty pages:", n_lines.count(0), "of", len(n_lines))
for sid, pages in preds.items():
    spk = sorted({l['speaker'] for pg in pages for l in pg})
    print(sid[:12], "|", spk, "|", [len(pg) for pg in pages])

print("\nPACKAGE:")
for root, _, files in os.walk(PKG):
    for fn in files:
        p = os.path.join(root, fn)
        print(p.replace(PKG, ''), round(os.path.getsize(p) / 1e6, 2), "MB")